In [1]:

# Import required libraries
import ollama
import time


# Select local LLM model
MODEL_NAME = "llama3.2"


In [2]:
def ensure_model_available(model_name: str):
    """Check that the model exists locally; pull it if it doesn't."""
    local_models = [m["name"] if "name" in m else m["model"] for m in ollama.list().get("models", [])]
    if not any(model_name in m for m in local_models):
        print(f"Model '{model_name}' not found locally. Pulling it now...")
        ollama.pull(model_name)
        print("Pull complete.")
    else:
        print(f"Model '{model_name}' is available locally.")

ensure_model_available(MODEL_NAME)

Model 'llama3.2' is available locally.


In [3]:
def generate_response(prompt: str, model: str = MODEL_NAME, temperature: float = 0.7):
    """Send a prompt to the local LLM and return the response text + stats."""
    start = time.time()
    result = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": temperature},
    )
    elapsed = time.time() - start

    reply_text = result["message"]["content"]
    stats = {
        "model": model,
        "temperature": temperature,
        "time_seconds": round(elapsed, 2),
        "prompt_tokens": result.get("prompt_eval_count"),
        "response_tokens": result.get("eval_count"),
    }
    return reply_text, stats

In [4]:
user_prompt = input("Enter your prompt: ")

reply, stats = generate_response(user_prompt)

print("\n--- PROMPT ---")
print(user_prompt)
print("\n--- RESPONSE ---")
print(reply)
print("\n--- STATS ---")
print(stats)

Enter your prompt:  Explain Artificial Intelligence in simple words.



--- PROMPT ---
Explain Artificial Intelligence in simple words.

--- RESPONSE ---
Artificial Intelligence (AI) is a way to create computers that can think and learn like humans.

Imagine you're trying to teach a child a new skill, like riding a bike. You start by showing them the basics, then gradually increase the difficulty as they get better. AI works in a similar way:

1. **Learning**: Computers with AI are programmed to learn from data, just like how we learn from our experiences.
2. **Pattern recognition**: They can identify patterns and relationships between things, which helps them make decisions or predictions.
3. **Improvement**: As they learn more, they get better at their tasks and can adapt to new situations.

There are different types of AI:

* **Narrow or Weak AI**: Good at specific tasks, like playing chess or recognizing faces (e.g., Siri, Alexa).
* **General or Strong AI**: Able to perform any intellectual task that a human can, like solving complex math problems or 

In [5]:
test_prompts = [
    "Explain Artificial Intelligence in simple words.",
    "Write a four-line poem about technology.",
    "If a student studies for 3 hours every day for 7 days, how many hours did the student study? Explain your calculation."
]
results = []
for p in test_prompts:
    reply, stats = generate_response(p)
    results.append({"prompt": p, "response": reply, "stats": stats})
    print(f"PROMPT: {p}")
    print(f"RESPONSE: {reply}")
    print(f"STATS: {stats}")
    print("-" * 80)

PROMPT: Explain Artificial Intelligence in simple words.
RESPONSE: Artificial Intelligence (AI) is a type of computer technology that allows machines to think and learn like humans.

Imagine you're teaching a child how to recognize pictures of animals. You show them many pictures of different animals, say "this is a dog" or "this is a cat", and they start to learn what makes each animal unique. Over time, the child becomes really good at recognizing animals just by looking at pictures.

AI works in a similar way. It's trained on huge amounts of data (like pictures) that teaches it how to recognize patterns, make decisions, and solve problems. The more data it receives, the smarter it becomes!

There are many types of AI, including:

1. Machine Learning: This is like the child learning to recognize animals. Machines learn from data and get better over time.
2. Natural Language Processing (NLP): This helps machines understand human language and communicate with us in a more natural way.


In [6]:
# Compare the same prompt at different temperatures (low = focused/deterministic, high = more random/creative)
temperature_prompt = "Describe a rainy day in one sentence."

for temp in [0.0, 0.9]:
    reply, stats = generate_response(temperature_prompt, temperature=temp)
    print(f"Temperature = {temp}")
    print(f"RESPONSE: {reply}")
    print(f"STATS: {stats}")
    print("-" * 80)

Temperature = 0.0
RESPONSE: The rain poured down from the grey sky, casting a soothing melody of droplets on the pavement and umbrellas, as the world outside seemed to slow down and wrap itself in a cozy, calming blanket of moisture.
STATS: {'model': 'llama3.2', 'temperature': 0.0, 'time_seconds': 3.66, 'prompt_tokens': 33, 'response_tokens': 44}
--------------------------------------------------------------------------------
Temperature = 0.9
RESPONSE: A rainy day is characterized by grey skies, heavy droplets of rain that fall from the sky, and a cool, damp air that seeps into every nook and cranny, creating a cozy and introspective atmosphere.
STATS: {'model': 'llama3.2', 'temperature': 0.9, 'time_seconds': 3.77, 'prompt_tokens': 33, 'response_tokens': 46}
--------------------------------------------------------------------------------


In [7]:
import requests

def generate_response_rest(prompt: str, model: str = MODEL_NAME):
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
        },
    )
    data = response.json()
    return data["message"]["content"]

# Example:
# print(generate_response_rest("Explain what an LLM is in one sentence."))